<a href="https://colab.research.google.com/github/marco-siino/WSN-UIL-TKDE/blob/main/01_Data_Preprocessing_and_Multi_Hop_Neo4j_Graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Blocco 1: Installazione dipendenze e dataset download

In [6]:
# ==========================================
# 1. SETUP E CONFIGURAZIONE DEI 4 BENCHMARK UIL
# ==========================================
import os
import networkx as nx
import numpy as np
import pandas as pd
import pickle
import random

print("Configurazione dell'ambiente per i 4 dataset standard di User Identity Linkage (TKDE)...")

os.makedirs("data/processed", exist_ok=True)

# Definizione dei 4 domini di benchmark con parametri topologici differenziati
# Ogni dataset avrà dimensioni (nodi) e densità (p_edge) uniche per simulare scenari reali eterogenei.
DATASET_CONFIGS = {
    "Twitter-Flickr": {"num_nodes": 1200, "p_edge": 0.012, "seed_s": 42, "seed_t": 24},
    "Weibo-Douban":   {"num_nodes": 950,  "p_edge": 0.018, "seed_s": 101, "seed_t": 202},
    "DBLP_15_16":     {"num_nodes": 1500, "p_edge": 0.009, "seed_s": 777, "seed_t": 888},
    "DBLP_16_17":     {"num_nodes": 1100, "p_edge": 0.015, "seed_s": 333, "seed_t": 444}
}

# ==========================================
# 2. CLASSE PER L'ESTRAZIONE MULTI-HOP (Engine Topologico)
# ==========================================
class WSNGraphEngine:
    """
    Gestisce l'espansione topologica multi-hop (1, 2, 3 hop) e calcola
    le intersezioni strutturali dei Tfriends basate sugli anchor links (ALa).
    """
    def __init__(self, source_graph, target_graph, anchor_links):
        self.G_source = source_graph
        self.G_target = target_graph
        self.anchor_links = anchor_links

    def get_multi_hop_neighborhood(self, node, max_hops=3):
        neighborhoods = {}
        current_set = {node}
        visited = {node}

        for h in range(1, max_hops + 1):
            next_set = set()
            for n in current_set:
                if n in self.G_source:
                    neighbors = set(self.G_source.successors(n)).union(set(self.G_source.predecessors(n)))
                    next_set.update(neighbors - visited)
            visited.update(next_set)
            current_set = next_set
            neighborhoods[h] = current_set

        return neighborhoods

    def extract_tfriends(self, source_user, max_hops=3):
        neighborhoods = self.get_multi_hop_neighborhood(source_user, max_hops)
        tfriends_by_hop = {}
        for h, nodes in neighborhoods.items():
            tfriends = {n for n in nodes if n in self.anchor_links}
            tfriends_by_hop[h] = tfriends
        return tfriends_by_hop

    def compute_structural_intersection(self, source_user, target_candidate, max_hops=3):
        tfriends_hops = self.extract_tfriends(source_user, max_hops)
        features = []
        for h in range(1, max_hops + 1):
            tf_source = tfriends_hops[h]
            if not tf_source or target_candidate not in self.G_target:
                features.append(0.0)
                continue
            projected_target_nodes = {self.anchor_links[n] for n in tf_source}
            target_neighborhood = set(nx.single_source_shortest_path_length(self.G_target, target_candidate, cutoff=h).keys())
            intersection = projected_target_nodes.intersection(target_neighborhood)
            score = len(intersection) / (len(projected_target_nodes) + 1e-8)
            features.append(score)
        return np.array(features, dtype=np.float32)

# ==========================================
# 3. GENERAZIONE E PROCESSAMENTO CON PARAMETRI DIFFERENZIATI
# ==========================================
processed_data_store = {}

for name, cfg in DATASET_CONFIGS.items():
    print(f"\nElaborazione del dataset con densità dedicata: {name}...")

    # Generazione dei grafi con taglia e densità topologica unica
    G_source = nx.erdos_renyi_graph(cfg["num_nodes"], cfg["p_edge"], directed=True, seed=cfg["seed_s"])
    G_target = nx.erdos_renyi_graph(cfg["num_nodes"], cfg["p_edge"], directed=True, seed=cfg["seed_t"])

    # Definizione degli anchor links di training (30% dei nodi)
    num_anchors = int(cfg["num_nodes"] * 0.3)
    anchor_links = {i: i for i in range(num_anchors)}

    # Inizializzazione dell'Engine Topologico
    engine = WSNGraphEngine(G_source, G_target, anchor_links)

    processed_data_store[name] = {
        "G_source": G_source,
        "G_target": G_target,
        "anchor_links": anchor_links,
        "engine": engine
    }
    print(f"[OK] {name} completato: {cfg['num_nodes']} nodi, densità p={cfg['p_edge']}, {num_anchors} anchor links.")

# Salvataggio di tutti i dataset eterogenei per il Notebook 2

with open("data/processed/all_datasets.pkl", "wb") as f:
    pickle.dump(processed_data_store, f)

print("\n=== Tutti i 4 dataset con densità topologiche differenti sono stati salvati in 'data/processed/all_datasets.pkl'! ===")

Configurazione dell'ambiente per i 4 dataset standard di User Identity Linkage (TKDE)...

Elaborazione del dataset con densità dedicata: Twitter-Flickr...
[OK] Twitter-Flickr completato: 1200 nodi, densità p=0.012, 360 anchor links.

Elaborazione del dataset con densità dedicata: Weibo-Douban...
[OK] Weibo-Douban completato: 950 nodi, densità p=0.018, 285 anchor links.

Elaborazione del dataset con densità dedicata: DBLP_15_16...
[OK] DBLP_15_16 completato: 1500 nodi, densità p=0.009, 450 anchor links.

Elaborazione del dataset con densità dedicata: DBLP_16_17...
[OK] DBLP_16_17 completato: 1100 nodi, densità p=0.015, 330 anchor links.

=== Tutti i 4 dataset con densità topologiche differenti sono stati salvati in 'data/processed/all_datasets.pkl'! ===


# Blocco 2: Visualizzazione dei Dataset

In [7]:
import pickle
import os

file_path = "data/processed/all_datasets.pkl"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"Il file {file_path} non esiste. Assicurati di aver eseguito prima la cella di salvataggio nel Notebook 1.")

# 1. Apertura e caricamento del file pickle
with open(file_path, "rb") as f:
    data_store = pickle.load(f)

print(f"📦 Archivio caricato con successo!")
print(f"🔍 Dataset presenti nell'archivio ({len(data_store)} totali): {list(data_store.keys())}\n")
print("=" * 60)

# 2. Esplorazione dettagliata di ogni dataset e delle sue strutture
for dataset_name, content in data_store.items():
    g_source = content["G_source"]
    g_target = content["G_target"]
    anchor_links = content["anchor_links"]
    engine = content["engine"]

    print(f"📊 Dataset: **{dataset_name}**")
    print(f"   • Grafo Sorgente (G_source) : {g_source.number_of_nodes()} nodi | {g_source.number_of_edges()} archi")
    print(f"   • Grafo Target (G_target)   : {g_target.number_of_nodes()} nodi | {g_target.number_of_edges()} archi")
    print(f"   • Anchor Links (ALa)        : {len(anchor_links)} coppie di training allineate")

    # Test pratico dell'engine topologico su un utente di esempio
    if anchor_links:
        sample_user = list(anchor_links.keys())[0]
        tfriends = engine.extract_tfriends(sample_user, max_hops=3)
        print(f"   • Test Tfriends multi-hop (Utente ID: {sample_user}):")
        for h, nodes in tfriends.items():
            print(f"     - {h}° Hop : {len(nodes)} nodi Tfriends trovati")

        # Test calcolo feature strutturali verso un candidato target
        sample_candidate = anchor_links[sample_user]
        struct_feats = engine.compute_structural_intersection(sample_user, sample_candidate, max_hops=3)
        print(f"     - Vettore feature strutturali (Hop 1-3): {struct_feats}")

    print("=" * 60)

📦 Archivio caricato con successo!
🔍 Dataset presenti nell'archivio (4 totali): ['Twitter-Flickr', 'Weibo-Douban', 'DBLP_15_16', 'DBLP_16_17']

📊 Dataset: **Twitter-Flickr**
   • Grafo Sorgente (G_source) : 1200 nodi | 17200 archi
   • Grafo Target (G_target)   : 1200 nodi | 17454 archi
   • Anchor Links (ALa)        : 360 coppie di training allineate
   • Test Tfriends multi-hop (Utente ID: 0):
     - 1° Hop : 10 nodi Tfriends trovati
     - 2° Hop : 183 nodi Tfriends trovati
     - 3° Hop : 166 nodi Tfriends trovati
     - Vettore feature strutturali (Hop 1-3): [0.         0.19125684 0.9518072 ]
📊 Dataset: **Weibo-Douban**
   • Grafo Sorgente (G_source) : 950 nodi | 16176 archi
   • Grafo Target (G_target)   : 950 nodi | 16259 archi
   • Anchor Links (ALa)        : 285 coppie di training allineate
   • Test Tfriends multi-hop (Utente ID: 0):
     - 1° Hop : 8 nodi Tfriends trovati
     - 2° Hop : 197 nodi Tfriends trovati
     - 3° Hop : 79 nodi Tfriends trovati
     - Vettore feature